In [40]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler
import math

In [41]:
df = pd.read_csv("final_dataset_hourly.csv", index_col=0, parse_dates=True)

# 填充 UVI > 20 为 NaN
df.loc[df["UVI"] > 20, "UVI"] = np.nan
# 线性插值补全
df = df.interpolate(method="linear")
# 夜间缺失的可填 0
df["UVI"].fillna(0, inplace=True)

# 周期特征编码
df['hour_sin'] = np.sin(2*np.pi*df.index.hour/24)
df['hour_cos'] = np.cos(2*np.pi*df.index.hour/24)
df['day_year_sin'] = np.sin(2*np.pi*df.index.dayofyear/365)
df['day_year_cos'] = np.cos(2*np.pi*df.index.dayofyear/365)

feature_cols = ['temp','rainfall','windspd','windspd_max','wind_d',
                'GHI','DNI','DHI','UVA','ClearSkyGHI','CSI','UVI',
                'hour_sin','hour_cos','day_year_sin','day_year_cos']
target_col = 'UVI'

# 数据集切分
train_df = df.loc['2020-01-01':'2021-06-30']
val_df   = df.loc['2021-07-01':'2021-09-30']
test_df  = df.loc['2021-10-01':'2021-12-29']

C:\Users\apple\AppData\Local\Temp\ipykernel_16784\2944346577.py:8: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df["UVI"].fillna(0, inplace=True)


In [12]:
x_scaler = StandardScaler()
y_scaler = StandardScaler()

train_x = x_scaler.fit_transform(train_df[feature_cols])
val_x   = x_scaler.transform(val_df[feature_cols])
test_x  = x_scaler.transform(test_df[feature_cols])

train_y = y_scaler.fit_transform(train_df[[target_col]])
val_y   = y_scaler.transform(val_df[[target_col]])
test_y  = y_scaler.transform(test_df[[target_col]])

In [13]:
import torch
from torch.utils.data import Dataset

class WeatherTimeSeriesDataset(Dataset):
    def __init__(self, data_matrix, target_matrix, seq_len=96, pred_len=24):
        self.seq_len = seq_len
        self.pred_len = pred_len
        self.valid_length = len(data_matrix) - seq_len - pred_len + 1
        assert self.valid_length > 0, "数据长度不足"

        self.x = torch.FloatTensor(data_matrix)
        self.y = torch.FloatTensor(target_matrix)

    def __len__(self):
        return self.valid_length

    def __getitem__(self, idx):
        seq_x = self.x[idx : idx + self.seq_len]
        seq_y = self.y[idx + self.seq_len : idx + self.seq_len + self.pred_len, 0]
        return seq_x, seq_y

In [14]:
train_dataset = WeatherTimeSeriesDataset(
    train_x,
    train_y,
    seq_len=96,
    pred_len=24
)

val_dataset = WeatherTimeSeriesDataset(
    val_x,
    val_y,
    seq_len=96,
    pred_len=24
)

test_dataset = WeatherTimeSeriesDataset(
    test_x,
    test_y,
    seq_len=96,
    pred_len=24
)

print(len(train_dataset))
print(len(val_dataset))
print(len(test_dataset))

13004
2089
2022


In [15]:
from torch.utils.data import DataLoader

train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=False
)

val_loader = DataLoader(
    val_dataset,
    batch_size=32,
    shuffle=False
)

test_loader = DataLoader(
    test_dataset,
    batch_size=32,
    shuffle=False
)

print(len(train_loader))
print(len(val_loader))
print(len(test_loader))

407
66
64


In [16]:
import torch
import torch.nn as nn

class PFLinear(nn.Module):

    def __init__(self,
                 seq_len=96,
                 pred_len=24,
                 n_features=16,
                 uvi_idx=9):   # 先确认UVI位置

        super().__init__()

        self.seq_len = seq_len
        self.pred_len = pred_len
        self.uvi_idx = uvi_idx

        # Linear分支
        self.dlinear = nn.Linear(seq_len, pred_len)

        # Gate
        self.gate = nn.Sequential(
            nn.Linear(n_features, 32),
            nn.ReLU(),
            nn.Linear(32, 1),
            nn.Sigmoid()
        )

    def forward(self, x):

        # --------------------
        # Linear Branch
        # --------------------
        uvi_hist = x[:, :, self.uvi_idx]      # [B,96]

        d_pred = self.dlinear(uvi_hist)       # [B,24]

        # --------------------
        # Persistence Branch
        # --------------------
        p_pred = uvi_hist[:, -24:]            # [B,24]

        # --------------------
        # Gate
        # --------------------
        feat = x.mean(dim=1)                  # [B,16]

        g = self.gate(feat)                   # [B,1]

        # --------------------
        # Fusion
        # --------------------
        out = g * d_pred + (1-g) * p_pred

        return out

In [17]:
UVI_idx = feature_cols.index("UVI")

print(UVI_idx)

11


In [19]:
import torch

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print(device)

cpu


In [20]:
model = PFDLinear(
    seq_len=96,
    pred_len=24,
    n_features=len(feature_cols),
    uvi_idx=UVI_idx
).to(device)

print(
    f"Params: {sum(p.numel() for p in model.parameters()):,}"
)

Params: 2,905


In [21]:
criterion = nn.MSELoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=1e-3
)

In [42]:
best_val = float("inf")
patience = 10
counter = 0

for epoch in range(50):

    model.train()

    train_loss = 0

    for x_batch, y_batch in train_loader:

        x_batch = x_batch.to(device)
        y_batch = y_batch.to(device)

        optimizer.zero_grad()

        pred = model(x_batch)

        loss = criterion(pred, y_batch)

        loss.backward()

        optimizer.step()

        train_loss += loss.item()

    train_loss /= len(train_loader)

    # Validation
    model.eval()

    val_loss = 0

    with torch.no_grad():

        for x_batch, y_batch in val_loader:

            x_batch = x_batch.to(device)
            y_batch = y_batch.to(device)

            pred = model(x_batch)

            loss = criterion(pred, y_batch)

            val_loss += loss.item()

    val_loss /= len(val_loader)

    print(
        f"Epoch {epoch+1:02d} | "
        f"Train={train_loss:.4f} | "
        f"Val={val_loss:.4f}"
    )

    if val_loss < best_val:

        best_val = val_loss

        torch.save(
            model.state_dict(),
            "best_pfdlinear.pt"
        )

        counter = 0

    else:

        counter += 1

        if counter >= patience:

            print(
                f"Early stopping "
                f"(best={best_val:.4f})"
            )
            break

Epoch 01 | Train=0.1786 | Val=0.3668
Epoch 02 | Train=0.1783 | Val=0.3669
Epoch 03 | Train=0.1780 | Val=0.3669
Epoch 04 | Train=0.1777 | Val=0.3669
Epoch 05 | Train=0.1774 | Val=0.3671
Epoch 06 | Train=0.1772 | Val=0.3674
Epoch 07 | Train=0.1769 | Val=0.3679
Epoch 08 | Train=0.1767 | Val=0.3685
Epoch 09 | Train=0.1764 | Val=0.3690
Epoch 10 | Train=0.1762 | Val=0.3695
Epoch 11 | Train=0.1760 | Val=0.3701
Early stopping (best=0.3668)


In [43]:
model.load_state_dict(
    torch.load("best_pfdlinear.pt")
)

model.eval()

preds = []
trues = []

with torch.no_grad():

    for x_batch, y_batch in test_loader:

        x_batch = x_batch.to(device)

        pred = model(x_batch)

        preds.append(
            pred.cpu().numpy()
        )

        trues.append(
            y_batch.numpy()
        )

In [44]:
import numpy as np

preds = np.concatenate(preds, axis=0)
trues = np.concatenate(trues, axis=0)

print(preds.shape)
print(trues.shape)

(2022, 24)
(2022, 24)


In [45]:
preds_inv = y_scaler.inverse_transform(preds)
trues_inv = y_scaler.inverse_transform(trues)

In [46]:
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

mae = mean_absolute_error(
    trues_inv.flatten(),
    preds_inv.flatten()
)

rmse = np.sqrt(
    mean_squared_error(
        trues_inv.flatten(),
        preds_inv.flatten()
    )
)

r2 = r2_score(
    trues_inv.flatten(),
    preds_inv.flatten()
)

print("PF-Linear Test")
print(f"MAE  = {mae:.4f}")
print(f"RMSE = {rmse:.4f}")
print(f"R²   = {r2:.4f}")

PF-Linear Test
MAE  = 0.3866
RMSE = 0.7297
R²   = 0.8259


In [47]:
time_index = []

for i in range(len(preds_inv)):
    start = test_df.index[i + 96]
    rng = pd.date_range(
        start=start,
        periods=24,
        freq='H'
    )
    time_index.extend(rng)

len(time_index)

C:\Users\apple\AppData\Local\Temp\ipykernel_16784\59912197.py:5: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  rng = pd.date_range(


48528

In [48]:
export_df = pd.DataFrame({
    "time": time_index,
    "True_UVI": trues_inv.flatten(),
    "UVCastNet_Pred": preds_inv.flatten()
})

export_df.to_csv(
    "UVCastNet_with_time.csv",
    index=False
)